# EEEM030 Group Assignment 2: Speech Recognition

Isolated-word recognition (11 words) with an FFN and a Transformer.

**Team members / chapter authors:** _TBC_

## 0. Setup

### 0.1 Imports

### 0.2 Environment specification & random seeds

### 0.3 Global hyper-parameters / config

## Part 2: Data Preparation & Feed-Forward Network

### 2.1 Load speech recordings & parse labels

### 2.2 Log-mel spectrogram extraction (80 mel bins, 25 ms window, 10 ms hop)

### 2.3 Fixed-length representation (pooling / pad-truncate / sequence)

### 2.4 Train / validation / test split (speaker-independent)

### 2.5 Plot example mel-spectrograms with labels

### 2.6 MLP from scratch (NumPy): forward pass

### 2.7 MLP in PyTorch (`nn.Module`)

### 2.8 Worked numerical check: NumPy vs PyTorch outputs

### 2.9 Shape diagram / dimension counting

## Part 3: Training with Backpropagation & SGD

### 3.1 NumPy MLP: backpropagation by hand

#### 3.1.1 Gradient derivation (cross-entropy + softmax, chain rule)

#### 3.1.2 Backward pass implementation

#### 3.1.3 Vanilla SGD / mini-batch training loop

#### 3.1.4 Loss & accuracy tracking (train / validation)

### 3.2 PyTorch MLP training

#### 3.2.1 Training with `torch.optim.SGD`

#### 3.2.2 Training with Adam

#### 3.2.3 Regularisation: weight decay & dropout

#### 3.2.4 Augmentation: SpecAugment / additive noise

### 3.3 Learning-rate schedules & early stopping

### 3.4 Results: loss & accuracy curves

### 3.5 Comparison table (accuracy, confusion matrix, parameters, training time)

### 3.6 Discussion: effect of dropout / SpecAugment on train–validation gap

## Part 4: Attention Mechanism

### 4.1 Scaled dot-product attention (from scratch)

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)V.
$$

In [ ]:
import math
## source: Tutorial_3_Attention_Mechanisms.ipynb
def scaled_dot_product_attention(q, k, v):
    """Calculate attention outputs and weights.

    Works with:
        One head:       (N, d_k).
        Batched heads:  (B, M, N, d_k).

    Values may have a different feature dimension, d_v.
    """
    d_k = q.shape[-1]

    # Compare every query with every key.
    scores = q @ k.transpose(-2, -1)  # (..., N, N).

    # Scale the scores.
    scores = scores / math.sqrt(d_k)

    # Convert scores into weights across the keys.
    weights = torch.softmax(scores, dim=-1)  # (..., N, N).

    # Combine the value vectors.
    output = weights @ v  # (..., N, d_v).

    return output, weights


### 4.2 Extend to multi-head attention
extended? meaning on top of the scaled_dot_product_attention?

In [ ]:
## source: https://docs.pytorch.org/docs/2.15/generated/torch.nn.MultiheadAttention.html
# multihead_attn = nn.MultiheadAttention(self.d_model, self.n_heads, batch_first = True) 
# attn_output, attn_output_weights = multihead_attn(q, k, v)
###### or just copy Tutorial_3_Attention_Mechanisms.ipynb: class MultiHeadSelfAttention(nn.Module):

### 4.3 Apply self-attention to a short sequence of mel frames (or the time dimension of a spectrogram), in order to visualise the attention weight matrix for a few utterances
- a short sequence of mel frames (do I need to pick, or the sequence is short already?)
- a few utterances? one batch/clip per utterance?
- does not say how many heads, I pressume n_head = 1

In [ ]:
# Create an untrained attention module with repeatable initial weights.
SEED 
D_MODEL = N_MELBIN
DEVICE
N_HEADS = 1
N_UTTER = BATCH_SIZE # assuming 1 batch for 1 utterance
NAME = 'Utterance'
x # Shape (batch/N_UTTER, n_melFrame, d_model/n_melBin)

# select utterance and melFrame for example utter:0~? & frame:20~60 
R_U = [0, N_UTTER]
R_F = [20, 60]
selected_x = x[R_F[0]:R_F[1], R_U[0]:R_U[1], :]
tokens = 

torch.manual_seed(SEED)
attention = nn.MultiheadAttention(
    embed_dim=D_MODEL,  # 128 features per token.
    num_heads=N_HEADS,  # 4 heads, each with 32 features.
    batch_first = True
).to(DEVICE)

attention.eval()

    # tokens contains the audio patch embeddings created earlier.
with torch.no_grad():
    attended_tokens, attention_weights = attention(selected_x, selected_x, selected_x, average_attn_weights=False)

    # print('Input tokens (B, N, D):', tuple(tokens.shape))
    # print('Output tokens (B, N, D):', tuple(attended_tokens.shape))
    # print(
    #     'Attention weights (B, heads, queries, keys):',
    #     tuple(attention_weights.shape),
    # )

    # # Each attention row should sum to one.
    # row_sums = attention_weights.sum(dim=-1)
    # torch.testing.assert_close(row_sums, torch.ones_like(row_sums))

    # print('\nVerified: each query’s attention weights sum to 1.')

# Select the first recording and retain all heads.
weights_to_plot = []
for i in range(0,N_UTTER):
    weights_to_plot[i] = attention_weights[i, 0].detach().cpu()
    # Shape: (N_HEADS, N_TOKENS, N_TOKENS).

fig, axes = plt.subplots(
    1, 1,
    figsize=(14, 4),
    layout='constrained',
    squeeze=False,
)

# Use the same colour scale across all frame.
minimum = weights_to_plot.min().item()
maximum = weights_to_plot.max().item()

# PLOT PER UTTERANCE GRAPH
for utter, ax in enumerate(axes[0]):
    image = ax.imshow(
        weights_to_plot[utter].numpy(),
        origin='upper',
        aspect='equal',
        cmap='Blues',
        vmin=minimum,
        vmax=maximum,
    )
    ax.set(
        title= NAME + f' {utter + 1}',
        xlabel='Key token index',
        ylabel='Query token index',
    )

fig.colorbar(
    image,
    ax=list(axes[0]),
    label='Attention weight',
    shrink=0.8,
)
fig.suptitle(
    'Untrained self-attention: one matrix per ' + NAME,
    fontweight='bold',
)
plt.show()

print(f'Observed weight range: {minimum:.5f} to {maximum:.5f}')
print(
    f'Uniform attention would assign '
    f'{1 / tokens.shape[1]:.5f} to each key.'
)

### 4.4 Worked example: take a 4–8 frame toy sequence, compute Q, K, V projections, attention weights and output by hand (or with detailed print statements)

In [ ]:
# pick batch/utter = 0, frame 20 to 23 I guess, give me an x of (1 utterance, 4 melframes, n_melbin) aka (4, D_MODEL)
x_toy
# step: 
# 1. q, k, v = x
# 2. follow def scaled_dot_product_attention(q, k, v):
# d_k = q.shape[-1]

# # Compare every query with every key.
# scores = q @ k.transpose(-2, -1)  # (..., N, N).

# # Scale the scores.
# scores = scores / math.sqrt(d_k)

# # Convert scores into weights across the keys.
# weights = torch.softmax(scores, dim=-1)  # (..., N, N).

# # Combine the value vectors.
# output = weights @ v  # (..., N, d_v).

# return output, weights


### 4.4.1 Verification against implementation

In [ ]:
# run
attention = nn.MultiheadAttention(
    embed_dim=D_MODEL,
    num_heads=1,
    batch_first=True
)
o_x_toy, w_x_toy = attention(x_toy, x_toy, x_toy, average_attn_weights=False)
# compare o_x_toy, w_x_toy with the hand cal

## Part 5: Basic Transformer Encoder for Mel-Spectrograms

### 5.1 Positional encodings (sinusoidal / learned)

### 5.2 Convolutional subsampling front-end (optional)

### 5.3 Transformer encoder block (MHSA + residual + LayerNorm, FFN + residual + LayerNorm)

### 5.4 Full model: 2 stacked encoder layers + classification head

### 5.5 Training & inference methods

### 5.6 Training curves & final test accuracy

### 5.7 Comparison vs MLP (accuracy & parameter count)

### 5.8 Analysis: does attention help? Effect of sequence length after subsampling

## Final Evaluation on Test Data

## Discussion Questions

### Why is the data-driven approach particularly powerful for speech?

### How does the √d_k scaling prevent vanishing gradients in the softmax?

### FFN on pooled features vs Transformer on full sequence: trade-offs

## Stretch Objectives (optional)

## References & Third-Party Code Attribution